# Demo C: a melody as a binary grid

Session 1, *Why encode music?*, part 9. The same demo runs on the course page; here you can change the inputs and keep your work.

The piano roll in Demo A is binary too: there is a hole at a position, or there is not. But a roll is an instruction for a machine: its time runs on continuously along the paper, and a long note is punched as a chain of holes. The grid in this notebook is something else: an array computed from a list of notes, with one row per pitch and one column per time step: 1 = the pitch sounds, 0 = it does not. CAMAT builds this kind of grid later in the course to search for patterns. We go from the melody to the grid and back, and see what survives.

**How to use this notebook:** run the cells from top to bottom with Shift+Enter. Change the values in the cells marked *Try this* and run them again.

## Setup

The first cell installs what the notebook needs; run it once after each server start. The second cell defines helper functions; it is collapsed because you do not need to read it.

In [ ]:
%pip install -q verovio==6.3.0 mido matplotlib ipywidgets

In [ ]:
# Helper functions for this notebook. Run this cell; you do not need to read it.
# They follow the same rules as the demos on the course page.

# Read pitches, metre and key from the MEI that Verovio writes.
#
# Why not Verovio's own MIDI values? Verovio 6.3 drops gestural accidentals
# when it imports **kern: with *k[f#c#], the token 4f# becomes
# <note pname="f"/> and getMIDIValuesForElement() reports F natural.
# So we apply the key signature ourselves:
#   accid.ges > accid > accidental earlier in the bar > key signature.

import re

STEPS = {"c": 0, "d": 2, "e": 4, "f": 5, "g": 7, "a": 9, "b": 11}
ACCID = {"s": 1, "f": -1, "ss": 2, "x": 2, "ff": -2, "n": 0, "xs": 3, "ts": 3, "tf": -3}
SHARP_ORDER = "fcgdaeb"
FLAT_ORDER = "beadgcf"


def _attrs(text):
    out = {}
    for m in re.finditer(r'([\w.:]+)="([^"]*)"', text, re.ASCII):
        out[m.group(1)] = m.group(2)
    return out


def _key_from_sig(sig):
    # "2s" -> {f: 1, c: 1}; "0" -> {}
    m = re.match(r"^(\d+)([sf]?)$", sig or "")
    key = {}
    if not m:
        return key
    n = int(m.group(1))
    order = FLAT_ORDER if m.group(2) == "f" else SHARP_ORDER
    for i in range(n):
        key[order[i]] = -1 if m.group(2) == "f" else 1
    return key


# Map xml:id -> MIDI pitch for every <note> with pname and oct.
def pitches_from_mei(mei):
    pitches = {}
    key = {}
    carried = {}  # "staff:pname:oct" -> alteration, reset per measure
    staff = "1"
    note = None  # open <note> waiting for a child <accid>

    def finish(n):
        where = f"{staff}:{n['pname']}:{n['oct']}"
        if n.get("ges") is not None:
            alter = n["ges"]
        elif n.get("written") is not None:
            alter = n["written"]
        elif where in carried:
            alter = carried[where]
        else:
            alter = key.get(n["pname"], 0)
        if n.get("written") is not None:
            carried[where] = n["written"]
        pitches[n["id"]] = 12 * (int(n["oct"]) + 1) + STEPS[n["pname"]] + alter

    for m in re.finditer(r"<(/?)([\w]+)\b([^>]*?)(/?)>", mei, re.ASCII):
        closing, tag, body, self_closing = m.group(1), m.group(2), m.group(3), m.group(4)
        if closing:
            if tag == "note" and note:
                finish(note)
                note = None
            continue
        a = _attrs(body)
        if "keysig" in a:
            key = _key_from_sig(a["keysig"])
        if tag == "keySig" and "sig" in a:
            key = _key_from_sig(a["sig"])
        if tag == "measure":
            carried = {}
        if tag == "staff" and a.get("n"):
            staff = a["n"]
        if tag == "note" and a.get("pname") and "oct" in a:
            n = {
                "id": a.get("xml:id"),
                "pname": a["pname"],
                "oct": a["oct"],
                "ges": ACCID.get(a.get("accid.ges")),
                "written": ACCID.get(a.get("accid")),
            }
            if self_closing:
                finish(n)
            else:
                note = n
        if tag == "accid" and note:
            if "accid.ges" in a:
                note["ges"] = ACCID.get(a["accid.ges"])
            if "accid" in a:
                note["written"] = ACCID.get(a["accid"])
    return pitches


# Metre {count, unit}; 4/4 if the MEI has none.
def meter_from_mei(mei):
    tag = re.search(r"<meterSig\b[^>]*>", mei, re.ASCII)
    source = tag.group(0) if tag else mei
    count = re.search(r'\b(?:meter\.)?count="(\d+)"', source, re.ASCII)
    unit = re.search(r'\b(?:meter\.)?unit="(\d+)"', source, re.ASCII)
    if count and unit:
        return {"count": int(count.group(1)), "unit": int(unit.group(1))}
    return {"count": 4, "unit": 4}


# First key signature as fifths: 2 = two sharps, -3 = three flats.
def fifths_from_mei(mei):
    m = re.search(r'\bkeysig="(\d+)([sf]?)"', mei, re.ASCII) or re.search(
        r'<keySig\b[^>]*\bsig="(\d+)([sf]?)"', mei, re.ASCII
    )
    if not m:
        return 0
    n = int(m.group(1))
    return -n if m.group(2) == "f" else n

# Melody <-> binary pitch x time grid (Demo C). Pure functions, no DOM.
# The Python version in 01_binary_grid.ipynb follows the same rules.
#
# A cell covers [i / res, (i + 1) / res) quarter notes. Two sets of rules:
# "demo": a note fills every cell whose start lies inside the note (what sounds
#   at the start of the cell); notes that cover no cell start are lost. The grid
#   is read back column by column, as a sequence of chords.
# "camat": as CAMAT's create_binary_matrix, onsets are floored and ends ceiled,
#   so a note fills every cell it touches. The grid is read back row by row, as
#   CAMAT's binary_slice_to_df does: each run of 1s is one note. CAMAT has no
#   onset layer (yet), so this read-back ignores onsets.

import math

NAMES = ["C", "C#", "D", "D#", "E", "F", "F#", "G", "G#", "A", "A#", "B"]


def pitch_name(midi):
    return NAMES[midi % 12] + str(math.floor(midi / 12) - 1)


def is_black_key(midi):
    return len(NAMES[midi % 12]) > 1


# Notes from a Verovio timemap: [{pitch, on, off}] in quarter notes.
# pitches: dict xml:id -> MIDI pitch (pitches_from_mei in mei.py); ids not in
# the map (rests) are skipped.
def notes_from_timemap(timemap, pitches):
    offs = {}
    for event in timemap:
        for id_ in event.get("off") or []:
            offs[id_] = event["qstamp"]
    notes = []
    for event in timemap:
        for id_ in event.get("on") or []:
            if id_ not in pitches:
                continue
            notes.append({
                "pitch": pitches[id_],
                "on": event["qstamp"],
                "off": offs.get(id_, event["qstamp"]),
            })
    notes.sort(key=lambda n: (n["on"], n["pitch"]))
    return notes


def bar_quarters(meter):
    return (meter["count"] * 4) / meter["unit"]


# Start of the cell grid: round to avoid 0.9999 artefacts.
EPS = 1e-9


def _js_round(x):
    # Math.round is round-half-up; Python round() is banker's rounding.
    return math.floor(x + 0.5)


# to_grid(notes, res, meter, rule) — positional (JS passes {res, meter, rule}).
def to_grid(notes, res, meter, rule="demo"):
    bar_cells = _js_round(bar_quarters(meter) * res)
    end = max([0] + [n["off"] for n in notes])
    bars = max(1, math.ceil(end / bar_quarters(meter) - EPS))
    n_cols = bars * bar_cells
    pitch_list = [n["pitch"] for n in notes]
    # Math.min(...[]) is Infinity; Math.max(...[]) is -Infinity.
    low = min(pitch_list) if pitch_list else float("inf")
    high = max(pitch_list) if pitch_list else float("-inf")
    occ = {}
    ons = {}
    p = low
    while p <= high:
        occ[p] = [0] * n_cols
        ons[p] = [0] * n_cols
        p += 1
    lost = []
    moved = []  # kept, but starting or ending at a different time
    for n in notes:
        if rule == "camat":
            first = math.floor(n["on"] * res + EPS)
        else:
            first = math.ceil(n["on"] * res - EPS)
        stop = math.ceil(n["off"] * res - EPS)  # first cell not covered
        if first >= stop or n["off"] - n["on"] < EPS:
            lost.append(n)
            continue
        if abs(first - n["on"] * res) > EPS or abs(stop - n["off"] * res) > EPS:
            moved.append(n)
        i = first
        while i < stop and i < n_cols:
            occ[n["pitch"]][i] = 1
            i += 1
        ons[n["pitch"]][first] = 1
    return {
        "res": res,
        "meter": meter,
        "rule": rule,
        "barCells": bar_cells,
        "nCols": n_cols,
        "low": low,
        "high": high,
        "occ": occ,
        "ons": ons,
        "lost": lost,
        "moved": moved,
    }


# Rows to show, highest first. compact: hide black-key rows that stay empty.
def grid_rows(grid, compact=True):
    rows = []
    p = grid["high"]
    while p >= grid["low"]:
        if compact and is_black_key(p) and not any(grid["occ"][p]):
            p -= 1
            continue
        rows.append(p)
        p -= 1
    return rows


# Grid as text, one row per pitch, cells grouped by bar (as on the slides).
def grid_to_text(grid, compact=True, onsets=False):
    rows = grid_rows(grid, compact)
    bars = grid["nCols"] / grid["barCells"]
    width = max(grid["barCells"], 5) + 2

    def label(s):
        return s.ljust(6)

    def block(cells):
        header = label("") + "".join(
            f"bar {b + 1}".ljust(width) for b in range(int(bars))
        ).rstrip()
        lines = [header]
        for p in rows:
            line = label(pitch_name(p))
            for b in range(int(bars)):
                chunk = cells[p][b * grid["barCells"]:(b + 1) * grid["barCells"]]
                line += "".join(str(v) for v in chunk).ljust(width)
            lines.append(line.rstrip())
        return "\n".join(lines)

    if not onsets:
        return block(grid["occ"])
    # Both grids in one: a note can only start where a pitch sounds, so three values are enough.
    three = {p: [(2 if grid["ons"][p][i] else 1) if v else 0 for i, v in enumerate(grid["occ"][p])] for p in rows}
    return (
        f"Sounding (1 = the pitch sounds):\n{block(grid['occ'])}\n\n"
        f"Onsets (1 = a note starts):\n{block(grid['ons'])}\n\n"
        f"Both in one grid (2 = a note starts, 1 = it continues, 0 = silence):\n{block(three)}"
    )


# Read the grid back as events: a new event starts whenever the set of
# sounding pitches changes or (with onsets) a sounding pitch has an onset.
def grid_to_events(grid, onsets=False):
    events = []
    current = None
    for i in range(grid["nCols"]):
        active = []
        attack = False
        p = grid["low"]
        while p <= grid["high"]:
            if grid["occ"][p][i]:
                active.append(p)
                if onsets and grid["ons"][p][i]:
                    attack = True
            p += 1
        key = ",".join(str(x) for x in active)
        if current and current["key"] == key and not attack:
            current["len"] += 1
        else:
            current = {"key": key, "start": i, "len": 1, "pitches": active}
            events.append(current)
    return [{"start": e["start"], "len": e["len"], "pitches": e["pitches"]} for e in events]


# Read the grid back with its rules: {events (for events_to_kern), notes, cut}.
# cut: how often the column-by-column reading cuts a held note because
# another pitch starts or stops while it sounds.
def read_grid(grid, onsets=False):
    if grid.get("rule") == "camat":
        notes = grid_to_notes(grid)
        return {"events": notes_to_events(notes, grid), "notes": notes, "cut": 0}
    events = grid_to_events(grid, onsets)
    cut = 0
    for k in range(1, len(events)):
        i = events[k]["start"]
        for p in events[k]["pitches"]:
            if p in events[k - 1]["pitches"] and not (onsets and grid["ons"][p][i]):
                cut += 1
    return {"events": events, "notes": events_to_notes(events, grid["res"]), "cut": cut}


# Row by row, as CAMAT reads its grids: each run of 1s in one row is one note,
# so repeated notes of the same pitch merge.
def grid_to_notes(grid):
    notes = []
    p = grid["low"]
    while p <= grid["high"]:
        start = -1
        for i in range(grid["nCols"] + 1):
            on = i < grid["nCols"] and grid["occ"][p][i] == 1
            if start >= 0 and not on:
                notes.append({"pitch": p, "on": start / grid["res"], "off": i / grid["res"]})
                start = -1
            if on and start < 0:
                start = i
        p += 1
    notes.sort(key=lambda n: (n["on"], n["pitch"]))
    return notes


# Notes -> events for events_to_kern: cut wherever a note starts or ends.
# tied: pitches that go on sounding from the event before.
def notes_to_events(notes, grid):
    res = grid["res"]
    spans = [(n["pitch"], _js_round(n["on"] * res), _js_round(n["off"] * res)) for n in notes]
    cuts = sorted({0, grid["nCols"], *(s[1] for s in spans), *(s[2] for s in spans)})
    events = []
    for k in range(len(cuts) - 1):
        here = [s for s in spans if s[1] <= cuts[k] < s[2]]
        events.append({
            "start": cuts[k],
            "len": cuts[k + 1] - cuts[k],
            "pitches": sorted(s[0] for s in here),
            "tied": sorted(s[0] for s in here if s[1] < cuts[k]),
        })
    return events


def events_to_notes(events, res):
    notes = []
    for e in events:
        for p in e["pitches"]:
            notes.append({
                "pitch": p,
                "on": e["start"] / res,
                "off": (e["start"] + e["len"]) / res,
            })
    notes.sort(key=lambda n: (n["on"], n["pitch"]))
    return notes


# kern pitch: c = C4, cc = C5, C = C3, CC = C2.
def kern_pitch(midi):
    name = NAMES[midi % 12]
    octave = math.floor(midi / 12) - 1
    letter = name[0]
    accidental = "#" if len(name) > 1 else ""
    repeat = octave - 3 if octave >= 4 else 4 - octave
    base = letter.lower() if octave >= 4 else letter
    return base * repeat + accidental


# Durations kern can write as one token, in quarter notes, largest first.
KERN_DURATIONS = [
    (6, "1."), (4, "1"), (3, "2."), (2, "2"), (1.5, "4."), (1, "4"),
    (0.75, "8."), (0.5, "8"), (0.375, "16."), (0.25, "16"), (0.125, "32"),
]


def _split_duration(quarters):
    parts = []
    left = quarters
    while left > EPS:
        fit = next(((q, recip) for q, recip in KERN_DURATIONS if q <= left + EPS), None)
        if not fit:
            break
        parts.append(fit)
        left -= fit[0]
    return parts


# Events (from grid_to_events or notes_to_events) -> a **kern score Verovio can render.
def events_to_kern(events, grid):
    res, meter, bar_cells = grid["res"], grid["meter"], grid["barCells"]
    sounding = [p for e in events for p in e["pitches"]]
    sorted_p = sorted(sounding)
    median = sorted_p[len(sorted_p) // 2] if sorted_p else 60
    lines = [
        "**kern",
        "*clefF4" if median < 57 else "*clefG2",
        f"*M{meter['count']}/{meter['unit']}",
    ]
    # Cut each event at bar lines, then into writable durations.
    pieces = []
    for k, e in enumerate(events):
        tied_in = e.get("tied", [])
        tied_out = events[k + 1].get("tied", []) if k + 1 < len(events) else []
        start = e["start"]
        end = e["start"] + e["len"]
        chunks = []
        while start < end:
            bar_end = (math.floor(start / bar_cells) + 1) * bar_cells
            stop = min(end, bar_end)
            for q, recip in _split_duration((stop - start) / res):
                chunks.append({"recip": recip, "start": start, "bar": math.floor(start / bar_cells)})
            start = stop
        for j, c in enumerate(chunks):
            pieces.append({**c, "pitches": e["pitches"], "first": j == 0, "last": j == len(chunks) - 1,
                           "tied_in": tied_in, "tied_out": tied_out})
    bar = -1
    for piece in pieces:
        if piece["bar"] != bar:
            bar = piece["bar"]
            lines.append(f"={bar + 1}")
        if not piece["pitches"]:
            lines.append(f"{piece['recip']}r")
            continue
        tokens = []
        for p in piece["pitches"]:
            token = piece["recip"] + kern_pitch(p)
            into = not piece["first"] or p in piece["tied_in"]
            onward = not piece["last"] or p in piece["tied_out"]
            if into and onward:
                tokens.append(token + "_")
            elif onward:
                tokens.append("[" + token)
            elif into:
                tokens.append(token + "]")
            else:
                tokens.append(token)
        lines.append(" ".join(tokens))
    lines.extend(["==", "*-"])
    return "\n".join(lines) + "\n"


# Compare two note lists: same pitches at the same times?
# Key by (pitch, on, off) so 1 and 1.0 match (JS string keys would differ).
def compare_notes(before, after):
    def key(n):
        return (n["pitch"], n["on"], n["off"])

    after_keys = {key(n) for n in after}
    before_keys = {key(n) for n in before}
    return {
        "kept": sum(1 for n in before if key(n) in after_keys),
        "missing": [n for n in before if key(n) not in after_keys],
        "added": [n for n in after if key(n) not in before_keys],
    }

# Input checks before Verovio sees the text, and melody checks for Demo D.
# Pure functions, no DOM. The notebooks carry the same rules in Python.

import re


# Some broken inputs crash Verovio 6.3 instead of giving an error, and some
# valid ones are misread (tested 2026-10-14). errors: do not load the text;
# warnings: load it, but say what Verovio will get wrong.
def precheck(fmt, text):
    problems = []
    warnings = []
    if fmt == "abc":
        lines = re.split(r"\r?\n", text)
        if not any(re.match(r"^X:", l) for l in lines):
            problems.append("ABC needs an X: line (for example X:1) before the tune.")
        if not any(re.match(r"^K:", l) for l in lines):
            problems.append("ABC needs a K: line (the key, for example K:C) before the notes.")
        depth = 0
        unmatched = False
        chord_length = False
        for line in lines:
            if re.match(r"^[A-Za-z]:", line):
                continue  # header and lyrics lines
            body = re.sub(r"%.*$", "", line)
            body = re.sub(r'"[^"]*"', "", body)
            body = re.sub(r"\|\]|\[\|", "|", body)
            if re.search(r"\][\d/]", body):
                chord_length = True
            for ch in body:
                if ch == "[":
                    depth += 1
                if ch == "]":
                    if depth == 0:
                        unmatched = True
                    else:
                        depth -= 1
        if unmatched:
            problems.append("There is a ] without a matching [ in the notes. Use |] for a final bar line.")
        if chord_length:
            warnings.append(
                "Verovio 6.3 ignores a length written after a chord, as in [CEG]2, and plays it as one unit. "
                "Write the length inside: [C2E2G2]."
            )
    if fmt == "humdrum":
        lines = [l for l in re.split(r"\r?\n", text) if l and not l.startswith("!!")]
        head = next((l for l in lines if l.startswith("**")), None)
        if head and re.search(r" {2,}|\*\*\w+ +\*\*", head, re.ASCII) and "\t" not in head:
            problems.append("Humdrum columns (spines) must be separated by tabs, not spaces.")
        manipulators = any(
            any(re.match(r"^\*[\^v+x]$", t) for t in l.split("\t"))
            for l in lines
        )
        if head and not manipulators:
            width = len(head.split("\t"))
            bad = next((i for i, l in enumerate(lines) if len(l.split("\t")) != width), -1)
            if bad >= 0:
                n_cols = len(lines[bad].split("\t"))
                problems.append(
                    f'Line "{lines[bad]}" has {n_cols} column(s), '
                    f"but the file has {width}. Every line needs one entry per spine."
                )
    return {"errors": problems, "warnings": warnings}


# LLM answers often come inside a Markdown code fence: keep only its content.
def clean_answer(text):
    fence = re.search(r"```[\w-]*\n([\s\S]*?)```", text, re.ASCII)
    return ((fence.group(1) if fence else text).strip() + "\n")


# Bar lengths in quarter notes, from the timemap's measure starts.
def bar_lengths(timemap, notes):
    starts = [e["qstamp"] for e in timemap if e.get("measureOn")]
    end = max(
        [0]
        + [e["qstamp"] for e in timemap]
        + [n["off"] for n in notes]
    )
    return [
        (starts[i + 1] if i + 1 < len(starts) else end) - s
        for i, s in enumerate(starts)
    ]


def parse_pitch(name):
    m = re.match(r"^([A-Ga-g])([#b]?)(-?\d)$", name.strip())
    if not m:
        return None
    step = {"c": 0, "d": 2, "e": 4, "f": 5, "g": 7, "a": 9, "b": 11}[m.group(1).lower()]
    alter = 1 if m.group(2) == "#" else -1 if m.group(2) == "b" else 0
    return 12 * (int(m.group(3)) + 1) + step + alter


def _js_num(n):
    # JS template strings write 2 for 2.0 and 3.5 for 3.5.
    if isinstance(n, float) and n.is_integer():
        return str(int(n))
    return str(n)


# Compare what Verovio understood with what the prompt asked for.
# found: {meter, fifths, bars: [lengths], notes}; spec: {meter, fifths, bars, low, high, final}
def melody_checks(found, spec):
    checks = []

    def add(label, ok, detail):
        checks.append({"label": label, "ok": ok, "detail": detail})

    want = (spec["meter"]["count"] * 4) / spec["meter"]["unit"]
    add(
        f"Metre is {spec['meter']['count']}/{spec['meter']['unit']}",
        found["meter"]["count"] == spec["meter"]["count"] and found["meter"]["unit"] == spec["meter"]["unit"],
        f"found {found['meter']['count']}/{found['meter']['unit']}",
    )
    add(f"{spec['bars']} bars", len(found["bars"]) == spec["bars"], f"found {len(found['bars'])}")
    wrong = [[i + 1, q] for i, q in enumerate(found["bars"]) if abs(q - want) > 1e-6]
    add(
        f"Every bar adds up to {_js_num(want)} quarter notes",
        len(wrong) == 0,
        ", ".join(f"bar {b}: {_js_num(q)}" for b, q in wrong) if wrong else "all bars complete",
    )

    def sig(f):
        if f == 0:
            return "no sharps or flats"
        if f > 0:
            return f"{f} sharp{'s' if f > 1 else ''}"
        return f"{-f} flat{'s' if f < -1 else ''}"

    add(f"Key signature: {sig(spec['fifths'])}", found["fifths"] == spec["fifths"], f"found {sig(found['fifths'])}")
    if found["notes"]:
        pitches = [n["pitch"] for n in found["notes"]]
        low = min(pitches)
        high = max(pitches)
        add(
            f"Range within {pitch_name(spec['low'])}–{pitch_name(spec['high'])}",
            low >= spec["low"] and high <= spec["high"],
            f"found {pitch_name(low)}–{pitch_name(high)}",
        )
        last = found["notes"][-1]["pitch"]
        final_name = re.sub(r"-?\d+$", "", pitch_name(spec["final"]))
        add(
            f"Ends on {final_name}",
            last % 12 == spec["final"] % 12,
            f"last note {pitch_name(last)}",
        )
    else:
        add("Has notes", False, "Verovio found no notes")
    return checks

# Verovio in a separate Python process.
#
# Some broken inputs crash Verovio 6.3 instead of giving an error (tested
# 2026-10-14): ABC without X: or K:, kern with too few columns. In a notebook
# a crash kills the kernel, and all your variables are gone. So Verovio runs
# in a child process: if it crashes, only the child dies. The child's error
# output is Verovio's log, which Jupyter would otherwise not show.

import json
import math
import os
import subprocess
import sys

from IPython.display import HTML, display

VEROVIO_VERSION = "6.3.0"

_CHILD = r'''
import json, sys
import verovio
job = json.loads(sys.stdin.read())
tk = verovio.toolkit()
tk.setOptions(job["options"])
out = {"ok": bool(tk.loadData(job["text"])), "version": tk.getVersion()}
if out["ok"]:
    out["svg"] = [tk.renderToSVG(p) for p in range(1, tk.getPageCount() + 1)] if job["svg"] else []
    out["mei"] = tk.getMEI()
    out["timemap"] = tk.renderToTimemap({"includeMeasures": True, "includeRests": True})
sys.stdout.write("\n@@RESULT@@\n" + json.dumps(out))
'''


# Load text in one format and return what the demos need (as on the web page).
# width: target width in pixels; Verovio lays the music out to fit it.
def read_score(fmt, text, width=800, scale=40, header="auto", svg=True):
    checked = precheck(fmt, text)
    result = {"ok": False, "errors": checked["errors"], "warnings": checked["warnings"],
              "log": "", "svg": [], "mei": "", "timemap": [], "notes": []}
    if result["errors"]:
        return result
    options = {
        "inputFrom": fmt, "scale": scale, "pageWidth": max(600, math.floor(width * 100 / scale + 0.5)),
        "adjustPageHeight": True, "pageMarginTop": 30, "pageMarginBottom": 30,
        "pageMarginLeft": 30, "pageMarginRight": 30, "header": header, "footer": "none",
    }
    child = subprocess.run(
        [sys.executable, "-c", _CHILD],
        input=json.dumps({"text": text, "options": options, "svg": svg}),
        capture_output=True, text=True, encoding="utf-8", timeout=300,
        env={**os.environ, "PYTHONIOENCODING": "utf-8"},
    )
    result["log"] = child.stderr.strip()
    if child.returncode < 0:
        result["errors"].append(
            f"Verovio crashed on this text (signal {-child.returncode}). This happens with some broken "
            "input that Verovio does not report as an error. Undo your last change and try again.")
        return result
    if child.returncode > 0 or "\n@@RESULT@@\n" not in child.stdout:
        result["errors"].append("The Verovio process failed; see the log below. Did the install cell run?")
        return result
    out = json.loads(child.stdout.rsplit("\n@@RESULT@@\n", 1)[1])
    if not out["ok"]:
        result["errors"].append(f"Verovio could not read this text as {fmt}.")
        return result
    result.update(ok=True, version=out["version"], svg=out["svg"], mei=out["mei"], timemap=out["timemap"])
    result["notes"] = notes_from_timemap(result["timemap"], pitches_from_mei(result["mei"]))
    result["meter"] = meter_from_mei(result["mei"])
    result["fifths"] = fifths_from_mei(result["mei"])
    return result


# The score on a white background (readable in dark themes), then the messages.
def show(result):
    for page in result["svg"]:
        display(HTML(f'<div style="background:white; padding:4px; overflow-x:auto">{page}</div>'))
    for text in result["errors"]:
        print("Error:", text)
    for text in result["warnings"]:
        print("Warning:", text)
    for line in result["log"].splitlines():
        if line.strip():
            print("Verovio:", line)


def render(fmt, text, **options):
    result = read_score(fmt, text, **options)
    show(result)
    return result

# Piano roll: time from left to right, pitch from bottom to top, one bar per note.
# The same view CAMAT uses later for its note tables.
#
# BACKEND chooses how rolls are drawn, as CAMAT's backend="plt" / "bokeh" does:
#   "plt"    a picture, drawn by matplotlib
#   "bokeh"  an interactive plot: drag to move, turn the mouse wheel to zoom,
#            point at a note to see its pitch, times and loudness.

import matplotlib.pyplot as plt
from matplotlib.collections import PatchCollection
from matplotlib.colors import to_rgba
from matplotlib.patches import Patch, Rectangle

BACKEND = "plt"
TRACK_COLORS = ["#1f77b4", "#d62728", "#9467bd", "#ff7f0e", "#2ca02c", "#8c564b"]
_NAMES = ["C", "C#", "D", "D#", "E", "F", "F#", "G", "G#", "A", "A#", "B"]
_UNITS = {"s": "time (seconds)", "q": "time (quarter notes)"}
_bokeh_loaded = False  # run this cell again after reloading the browser tab


def _pitch_label(p):
    return f"{_NAMES[p % 12]}{p // 12 - 1}"


def _color(n):
    return TRACK_COLORS[n.get("track", 0) % len(TRACK_COLORS)]


# notes: [{"pitch", "start", "end"}], optionally "track" (colour), "role" (legend),
#        "alpha" (opacity) and "velocity" (shown when you point at a note).
# bands: [{"start", "end", "row", "color", "label"}] drawn under the keyboard (pedals).
# bar: draw a line every `bar` time units (for scores: quarter notes per bar).
# backend: "plt" or "bokeh"; None uses BACKEND.
def draw_roll(notes, start=None, end=None, low=None, high=None, bands=(), bar=None,
              unit="s", title=None, width=12, height=None, backend=None):
    if start is None:
        start = min((n["start"] for n in notes), default=0)
    if end is None:
        end = max((n["end"] for n in notes), default=start + 1)
    pitches = [n["pitch"] for n in notes]
    if low is None:
        low = min(pitches) - 1 if pitches else 59
    if high is None:
        high = max(pitches) + 1 if pitches else 72
    rows = sorted({b["row"] for b in bands})
    if height is None:
        height = min(8, max(2.5, 0.12 * (high - low + 2 * len(rows)) + 1))
    backend = (backend or BACKEND).strip().lower()
    if backend == "plt":
        _roll_plt(notes, start, end, low, high, bands, rows, bar, unit, title, width, height)
    elif backend == "bokeh":
        _roll_bokeh(notes, start, end, low, high, bands, rows, bar, unit, title, height)
    else:
        raise ValueError(f"Unknown backend {backend!r}: use 'plt' or 'bokeh'.")


def _roll_plt(notes, start, end, low, high, bands, rows, bar, unit, title, width, height):
    below = 2 * len(rows)
    fig, ax = plt.subplots(figsize=(width, height))
    for p in range(low, high + 1):
        if p % 12 in (1, 3, 6, 8, 10):
            ax.axhspan(p - 0.5, p + 0.5, color="0.93", lw=0, zorder=0)
    shortest = (end - start) / 1500
    rects = [Rectangle((n["start"], n["pitch"] - 0.4), max(n["end"] - n["start"], shortest), 0.8) for n in notes]
    colors = [to_rgba(_color(n), n.get("alpha", 0.9)) for n in notes]
    ax.add_collection(PatchCollection(rects, facecolors=colors, edgecolors="none", zorder=2))
    ticks = [(p, _pitch_label(p)) for p in range(low, high + 1) if p % 12 == 0]
    for b in bands:
        y = low - 1.5 - 2 * rows.index(b["row"])
        ax.add_patch(Rectangle((b["start"], y - 0.6), b["end"] - b["start"], 1.2, color=b["color"], lw=0))
    for row in rows:
        label = next(b["label"] for b in bands if b["row"] == row)
        ticks.append((low - 1.5 - 2 * rows.index(row), label))
    if bar:
        t = bar * (start // bar)
        while t <= end:
            ax.axvline(t, color="0.6", lw=0.8, zorder=1)
            t += bar
    roles = {}
    for n in notes:
        if n.get("role"):
            roles.setdefault(n["role"], _color(n))
    if roles:
        ax.legend(handles=[Patch(color=c, label=r) for r, c in roles.items()], loc="upper left", fontsize=8)
    ax.set_yticks([y for y, _ in ticks], [label for _, label in ticks])
    ax.set_xlim(start, end)
    ax.set_ylim(low - 0.5 - below, high + 0.5)
    ax.set_xlabel(_UNITS.get(unit, unit))
    ax.set_ylabel("pitch")
    ax.grid(axis="y", color="0.85", lw=0.6)
    if title:
        ax.set_title(title)
    fig.tight_layout()
    plt.show()


def _roll_bokeh(notes, start, end, low, high, bands, rows, bar, unit, title, height):
    global _bokeh_loaded
    from bokeh.io import output_notebook, show
    from bokeh.models import ColumnDataSource, FixedTicker, HoverTool, Range1d, Span
    from bokeh.plotting import figure
    from bokeh.resources import INLINE

    if not _bokeh_loaded:
        output_notebook(resources=INLINE, hide_banner=True)  # Bokeh's JavaScript, once per page
        _bokeh_loaded = True
    below = 2 * len(rows)
    fig = figure(
        title=title, height=int(90 * height), sizing_mode="stretch_width",
        x_range=Range1d(start, end, bounds=(start, end)),
        y_range=Range1d(low - 0.5 - below, high + 0.5, bounds=(low - 0.5 - below, high + 0.5)),
        x_axis_label=_UNITS.get(unit, unit), y_axis_label="pitch",
        tools="xpan,xwheel_zoom,box_zoom,reset,save", active_drag="xpan", active_scroll="xwheel_zoom",
    )
    fig.xgrid.grid_line_color = None
    keys = [p for p in range(low, high + 1) if p % 12 in (1, 3, 6, 8, 10)]
    fig.hbar(y=keys, left=start, right=end, height=1, color="black", alpha=0.06)
    if bar:
        t = bar * (start // bar)
        while t <= end:
            fig.add_layout(Span(location=t, dimension="height", line_color="#999999", line_width=1))
            t += bar

    shortest = (end - start) / 1500
    groups = {}
    for n in notes:
        groups.setdefault((n.get("track", 0), n.get("role")), []).append(n)
    for (track, role), group in sorted(groups.items(), key=lambda item: item[0][0]):
        data = {
            "left": [n["start"] for n in group],
            "right": [max(n["end"], n["start"] + shortest) for n in group],
            "bottom": [n["pitch"] - 0.4 for n in group],
            "top": [n["pitch"] + 0.4 for n in group],
            "alpha": [n.get("alpha", 0.9) for n in group],
            "name": [_pitch_label(n["pitch"]) for n in group],
            "start": [n["start"] for n in group],
            "end": [n["end"] for n in group],
        }
        tips = [("pitch", "@name"), ("start", "@start{0.000}"), ("end", "@end{0.000}")]
        if all("velocity" in n for n in group):
            data["velocity"] = [n["velocity"] for n in group]
            tips.append(("velocity", "@velocity"))
        legend = {"legend_label": role} if role else {}
        glyph = fig.quad(source=ColumnDataSource(data), left="left", right="right", bottom="bottom", top="top",
                         fill_color=_color(group[0]), fill_alpha="alpha", line_color=None, **legend)
        fig.add_tools(HoverTool(renderers=[glyph], tooltips=tips))
    if fig.legend:
        fig.legend.location = "top_left"
        fig.legend.click_policy = "hide"  # click a track in the legend to hide it

    ticks = {p: _pitch_label(p) for p in range(low, high + 1) if p % 12 == 0}
    for b in bands:
        y = low - 1.5 - 2 * rows.index(b["row"])
        fig.quad(left=b["start"], right=b["end"], bottom=y - 0.6, top=y + 0.6, color=b["color"], line_color=None)
        ticks[y] = b["label"]
    fig.yaxis.ticker = FixedTicker(ticks=sorted(ticks))
    fig.yaxis.major_label_overrides = ticks
    show(fig)


# Piano roll of a score read by read_score(): time in quarter notes, a line per bar.
def score_roll(result, **options):
    notes = [{"pitch": n["pitch"], "start": n["on"], "end": n["off"]} for n in result["notes"]]
    meter = result.get("meter") or {"count": 4, "unit": 4}
    draw_roll(notes, start=0, bar=meter["count"] * 4 / meter["unit"], unit="q", **options)

# Demo C: draw the grid, change cells from code (on the web page you click
# them), and read the grid back as notes.

import matplotlib.pyplot as plt
import numpy as np

_EMPTY, _SOUND, _START, _CONT = (1, 1, 1), (0.29, 0.56, 0.85), (0.12, 0.37, 0.66), (0.65, 0.78, 0.93)


# One row per pitch (highest at the top), one column per cell, a line at each bar.
# With onsets: dark = a note starts, light = it continues (not with CAMAT's
# rules: CAMAT has no onset layer yet).
def draw_grid(grid, onsets=False, compact=True):
    onsets = onsets and grid["rule"] != "camat"
    rows = grid_rows(grid, compact)
    n = grid["nCols"]
    image = np.ones((len(rows), n, 3))
    for r, p in enumerate(rows):
        for i in range(n):
            if grid["occ"][p][i]:
                image[r, i] = (_START if grid["ons"][p][i] else _CONT) if onsets else _SOUND
    fig, ax = plt.subplots(figsize=(min(14, 1.2 + 0.32 * n), 0.6 + 0.32 * len(rows)))
    ax.imshow(image, aspect="auto", interpolation="nearest")
    if n <= 96:
        for r, p in enumerate(rows):
            for i in range(n):
                value = (2 if onsets and grid["ons"][p][i] else 1) if grid["occ"][p][i] else 0
                ax.text(i, r, value, ha="center", va="center", fontsize=8,
                        color="white" if grid["occ"][p][i] else "0.6")
    for x in np.arange(-0.5, n, 1):
        ax.axvline(x, color="0.85", lw=0.5)
    for b in range(0, n + 1, grid["barCells"]):
        ax.axvline(b - 0.5, color="black", lw=1.5)
    ax.set_yticks(range(len(rows)), [pitch_name(p) for p in rows])
    bars = n // grid["barCells"]
    ax.set_xticks([b * grid["barCells"] + (grid["barCells"] - 1) / 2 for b in range(bars)], [f"bar {b + 1}" for b in range(bars)])
    ax.tick_params(length=0)
    fig.tight_layout()
    plt.show()


# Change one cell, as a click on the web page does. Cells count from 1.
# value: 0 (silent), 1 (sounding) or 2 (sounding, and a note starts here; "start" works too).
def set_cell(grid, name, cell, value):
    pitch = parse_pitch(name)
    if pitch is None:
        raise ValueError(f"{name!r} is not a pitch: write for example G4, F#4 or Bb3")
    if not 1 <= cell <= grid["nCols"]:
        raise ValueError(f"The grid has cells 1 to {grid['nCols']}")
    for p in range(min(pitch, grid["low"]), max(pitch, grid["high"]) + 1):
        grid["occ"].setdefault(p, [0] * grid["nCols"])
        grid["ons"].setdefault(p, [0] * grid["nCols"])
    grid["low"] = min(grid["low"], pitch)
    grid["high"] = max(grid["high"], pitch)
    i = cell - 1
    if value == 0:
        grid["occ"][pitch][i] = grid["ons"][pitch][i] = 0
    elif value in (2, "start"):
        grid["occ"][pitch][i] = grid["ons"][pitch][i] = 1
    else:
        grid["occ"][pitch][i] = 1
        grid["ons"][pitch][i] = 0 if i > 0 and grid["occ"][pitch][i - 1] else 1


# Grid -> notes and **kern with the grid's rules, compared with the melody we started from.
def round_trip(source, grid, onsets=False):
    if onsets and grid["rule"] == "camat":
        print("CAMAT has no onset layer (yet), so rule=\"camat\" reads the grid without onsets.")
        onsets = False
    back = read_grid(grid, onsets=onsets)
    kern = events_to_kern(back["events"], grid)
    cmp = compare_notes(source["notes"], back["notes"])
    n = len(source["notes"])
    print(f"The melody has {n} notes. Read back from the grid: {len(back['notes'])} notes, {cmp['kept']} of them unchanged.")

    def names(notes):
        return ", ".join(pitch_name(x["pitch"]) for x in notes)

    if grid["lost"]:
        print(f"{len(grid['lost'])} note(s) start and end inside one cell and disappear: {names(grid['lost'])}. Try a smaller cell, or rule=\"camat\".")
    if grid["moved"]:
        print(f"{len(grid['moved'])} note(s) now start or end on a cell edge instead of where they were written: {names(grid['moved'])}.")
    if back["cut"]:
        print(f"Held notes are cut {back['cut']} time(s) where another pitch starts or stops: rule=\"demo\" reads the grid column by column, as chords. CAMAT's rules read each row on its own.")
    if cmp["kept"] == n and len(back["notes"]) == n:
        print("Nothing lost: the round trip gives back the same notes.")
    elif grid["rule"] == "camat":
        print("CAMAT reads a run of 1s as one held note, so repeated notes of the same pitch merge: CAMAT has no onset layer (yet).")
    elif not onsets:
        print("Without onsets, a run of 1s is read as one held note, so repeated notes of the same pitch merge. Use onsets=True to keep them apart.")
    render("humdrum", kern, scale=30, header="none")
    return kern

# Show sliders and menus (ipywidgets). Where the browser cannot draw them, Jupyter
# prints the widget as text ("interactive(children=(Dropdown(...") instead; this
# prints what to do.

from IPython.display import display


def show_controls(controls, by_hand):
    help_text = (
        "[The sliders and menus did not load. Reload the browser tab once (the install cell has "
        "just added the widget library) and run this cell again. Or skip the sliders and type, "
        f"in a new cell: {by_hand}]"
    )
    bundle = getattr(controls, "_repr_mimebundle_", None)
    if bundle is None:  # an older ipywidgets: show it the usual way
        display(controls)
        return
    data = bundle()
    data["text/plain"] = help_text
    display(data, raw=True)

## 1. The melody

*Alle meine Entchen* in ABC. Verovio reads it; `source["notes"]` is the list of notes it found, with start (`on`) and end (`off`) in quarter notes.

In [ ]:
melody = """X:1
T:Alle meine Entchen
M:4/4
L:1/4
K:C
C D E F | G2 G2 | A A A A | G4 |]
"""
source = render("abc", melody, scale=30, header="none")
for n in source["notes"]:
    print(f"{pitch_name(n['pitch']):4} from {n['on']:>4} to {n['off']:>4}")

## 2. The grid

`RES` is the number of cells per quarter note: 1 = one cell per quarter note, 2 = per eighth note, 4 = per sixteenth note.

`RULE` says which notes fill a cell:

- `"demo"`: what sounds at the start of the cell. A note fills every cell whose start lies inside the note.
- `"camat"`: what sounds at any time during the cell, as in CAMAT's `create_binary_matrix`: the start of each note is moved back and its end forward to the nearest cell edge.

For this melody both rules give the same grid; part 5 shows where they differ.

In [ ]:
RES = 1  # Try this: 2 or 4
RULE = "demo"  # Try this: "camat"
grid = to_grid(source["notes"], RES, source["meter"], RULE)
draw_grid(grid)
print(grid_to_text(grid))

*Which bars can you no longer tell apart?*

## 3. Back from the grid to notes

Reading the grid back needs a rule too. Both rules read a run of 1s in one row as one held note, so `1111` comes back as a whole note. `"demo"` reads the grid column by column, as a sequence of chords: a new note starts wherever the set of sounding pitches changes. `"camat"` reads each row on its own, as CAMAT's `binary_slice_to_df` does. For one melodic line the two agree; part 6 holds a note under the melody, where they do not. The helper writes the result as `**kern`, renders it, and compares it with the melody.

In [ ]:
kern = round_trip(source, grid)
print(kern)

Bars 2 (two half notes), 3 (four quarter notes) and 4 (one whole note) all become a run of four cells. Pure occupancy loses where one note ends and the next begins.

Why not read every 1 as a new note? That would give four quarter notes: right for bar 3, wrong for bars 2 and 4, and with `RES = 4` every note would come back as sixteenths. Without onsets, no rule gets all three bars right.

## 4. A second grid for onsets

One fix: keep a second grid that marks where notes start. In the picture, dark cells are note starts and light cells are continuations. CAMAT has no onset layer yet: it keeps the onsets in its note table, so `rule="camat"` ignores `onsets=True`.

The two grids together are one grid with three values: **0** = silence, **1** = a note continues, **2** = a note starts. Two yes/no grids would allow four combinations, but "a note starts where nothing sounds" cannot happen, so three are left: the grid is no longer binary but *ternary*. Every cell that is not 0 still means "sounding", so a tool written for the 0/1 grid can read it too. The last block printed below is this ternary grid.

In [ ]:
draw_grid(grid, onsets=True)
print(grid_to_text(grid, onsets=True))
kern = round_trip(source, grid, onsets=True)

## 5. Resolution is a choice

A melody with eighth notes, on a grid with one cell per quarter note. Here the two rules part ways:

- `"demo"` keeps only what sounds at the start of each cell. The eighth notes off the beat disappear, and those on the beat grow to quarter notes.
- `"camat"` loses nothing, but every note grows to fill whole cells, so two eighth notes in one cell sound together, as a chord.

Finer grids keep more, but cost size: CAMAT's default cell is a sixteenth note.

In [ ]:
melody2 = """X:1
T:With eighth notes
M:4/4
L:1/8
K:C
C D E F G2 G2 | A A A A G4 |]
"""
source2 = render("abc", melody2, scale=30, header="none")

RES2 = 1  # Try this: 2
for rule in ["demo", "camat"]:
    print(f"--- rule = {rule!r}, {RES2} cell(s) per quarter note")
    grid2 = to_grid(source2["notes"], RES2, source2["meter"], rule)
    onsets = rule == "demo"  # CAMAT has no onset layer yet
    draw_grid(grid2, onsets=onsets)
    round_trip(source2, grid2, onsets=onsets)

## 6. Change the grid

On the course page you click cells; here you call `set_cell(grid, pitch, cell, value)`. Cells count from 1; with one cell per quarter note, bar 2 is cells 5 to 8. The value is `0` (silent), `1` (sounding) or `2` (sounding, and a note starts here): the three values of the ternary grid.

In [ ]:
grid = to_grid(source["notes"], 1, source["meter"])
set_cell(grid, "G4", 8, 0)      # shorten the second G in bar 2
set_cell(grid, "B4", 13, 1)     # add a B4 at the start of bar 4
set_cell(grid, "G4", 13, 0)
draw_grid(grid)
kern = round_trip(source, grid)

Now hold a G3 under the first bar. `"demo"` reads column by column, so it cuts the G3 wherever the melody moves; `"camat"` reads the G3's row on its own and keeps one whole note.

In [ ]:
for rule in ["demo", "camat"]:
    print(f"--- rule = {rule!r}")
    grid = to_grid(source["notes"], 1, source["meter"], rule)
    for cell in range(1, 5):
        set_cell(grid, "G3", cell, 1)
    draw_grid(grid)
    round_trip(source, grid)

**Try this:** turn bar 3 (cells 9 to 12) into two half notes on A4. A `1` in a cell right after a sounding cell continues the note instead of starting a new one.

In [ ]:
grid = to_grid(source["notes"], 1, source["meter"])
# your set_cell(...) lines here
draw_grid(grid, onsets=True)
kern = round_trip(source, grid, onsets=True)

## 7. Explore with controls

Change the melody in part 1, then choose the cell size, the rules and whether to keep onsets (only with this demo's rules: CAMAT has no onset layer yet). If the controls do not appear, the cell says what to do.

In [ ]:
import ipywidgets as widgets

def explore(res=1, rule="demo", onsets=False):
    g = to_grid(source["notes"], res, source["meter"], rule)
    draw_grid(g, onsets=onsets)
    round_trip(source, g, onsets=onsets)

rule_menu = widgets.Dropdown(options=[("this demo's", "demo"), ("CAMAT's", "camat")], value="demo", description="rules")
onset_box = widgets.Checkbox(value=False, description="keep an onset grid")

def no_onsets_for_camat(change):  # CAMAT has no onset layer yet
    onset_box.disabled = change["new"] == "camat"
    if onset_box.disabled:
        onset_box.value = False

rule_menu.observe(no_onsets_for_camat, names="value")
controls = widgets.interactive(
    explore,
    res=widgets.Dropdown(options=[("quarter note", 1), ("eighth note", 2), ("sixteenth note", 4)], value=1, description="cell"),
    rule=rule_menu,
    onsets=onset_box,
)
show_controls(controls, 'explore(res=2, rule="demo", onsets=True)')

## 8. What CAMAT does with the grid

CAMAT does not read its grid back as the music. It slides a small grid, such as a motif, over the grid of a whole piece and scores each position by the cells they share. For every 1, CAMAT can keep a pointer to the notes in the score behind it, so a match is shown in the original score, not in a score rebuilt from 0s and 1s. Statistics such as how often each pitch occurs come from CAMAT's note table, not from the grid. CAMAT's notebooks [binary_roundtrip.ipynb](https://github.com/egorpol/camat_v2/blob/main/notebooks/binary_roundtrip.ipynb) and [binary_pattern_search.ipynb](https://github.com/egorpol/camat_v2/blob/main/notebooks/binary_pattern_search.ipynb) show both.

## Questions

- What does the grid record, and what does it leave out?
- What would you need to keep, next to the grid, to get the melody back exactly? CAMAT keeps a note table next to its grids, with identifiers that point back into the score.